# QubitGrid Krylov Analysis: Why P(unreachable) = 0

This notebook explains why the Krylov criterion always gives P=0 for QubitGrid models,
while Spectral and Moment show clear phase transitions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.models import QubitGridModel, CanonicalQuditModel

## The Puzzle

For QubitGrid models:
- **Spectral criterion**: Shows clear phase transitions
- **Moment criterion**: Shows transitions for small d
- **Krylov criterion**: Always P=0 (all states "reachable")

Why is Krylov uninformative for QubitGrid?

## Key Insight

The Krylov subspace is:
$$\mathcal{K}_m(H, \phi) = \text{span}\{\phi,\; H\phi,\; H^2\phi,\; \ldots,\; H^{m-1}\phi\}$$

For **generic** Hamiltonians (random Pauli sums), this subspace quickly spans the **entire Hilbert space**. When $\dim(\mathcal{K}_m) = d$, any target $\psi$ satisfies:
$$\|\text{Proj}_{\mathcal{K}} \psi\|^2 = \|\psi\|^2 = 1.0 \geq \tau$$

So ALL states are trivially "reachable" by the Krylov criterion.

## Demonstration: Two Pauli Operators Span Full Space

We'll show that combining just **two** generic Pauli operators creates a Hamiltonian whose Krylov subspace spans the entire Hilbert space.

**Setup:**
- System: 2 qubits (d = 4)
- Operators: $H_1 = X_1 \otimes Y_2$ and $H_2 = Z_1 \otimes X_2$
- Combined: $H = 0.7 \cdot H_1 + 0.3 \cdot H_2$

**What to look for:** The Krylov basis $\{\phi, H\phi, H^2\phi, H^3\phi\}$ should have dimension 4 (the full Hilbert space dimension).

In [ ]:
# Pauli matrices
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Two 2-local Pauli terms combined with random weights
H1 = np.kron(X, Y)  # X_1 x Y_2
H2 = np.kron(Z, X)  # Z_1 x X_2
H = 0.7 * H1 + 0.3 * H2

print("H = 0.7*(X_1 x Y_2) + 0.3*(Z_1 x X_2):")
print(np.round(H, 3))

In [ ]:
def build_krylov_basis(H, phi, m):
    """Build orthonormal Krylov basis via Gram-Schmidt."""
    d = len(phi)
    V = [phi / np.linalg.norm(phi)]
    for j in range(m - 1):
        w = H @ V[-1]
        for v in V:
            w = w - np.vdot(v, w) * v
        norm = np.linalg.norm(w)
        if norm < 1e-10:
            break
        V.append(w / norm)
    return np.column_stack(V)

phi = np.array([1, 0, 0, 0], dtype=complex)  # |00>
V = build_krylov_basis(H, phi, m=4)

print(f"Initial state: |00>")
print(f"Krylov basis dimension: {V.shape[1]}  (full space = {len(phi)})")
print(f"\nBasis vectors (columns):")
print(np.round(V, 3))
if V.shape[1] == len(phi):
    print(f"\n-> Just 2 Pauli operators span the entire d={len(phi)} space!")
else:
    print(f"\n-> Krylov dimension = {V.shape[1]} / {len(phi)}")

### Interpretation

The Krylov basis has **dimension 4 out of 4** -- it spans the entire Hilbert space!

This means for ANY target state $\psi$:
$$\|\text{Proj}_{\mathcal{K}} \psi\|^2 = \|\psi\|^2 = 1.0$$

Since our reachability threshold is $\tau = 0.99$, every target is classified as "reachable." **This is why P(unreachable) = 0 for QubitGrid.**

The combined Hamiltonian mixes all computational basis states. $X \otimes Y$ couples $|00\rangle \leftrightarrow |11\rangle$ and $|01\rangle \leftrightarrow |10\rangle$, while $Z \otimes X$ couples across these pairs. After just a few matrix-vector products, we reach all 4 basis states. **Generic Pauli operators are highly mixing** -- they don't preserve any subspace.

## Contrast: Canonical Basis

Canonical operators like $Z_j = |j\rangle\langle j| - |j{+}1\rangle\langle j{+}1|$ are **diagonal**. They map basis states to themselves, so the Krylov subspace stays small.

In [ ]:
# Z_0 = |0><0| - |1><1| for d=4
Z0 = np.diag([1, -1, 0, 0]).astype(complex)
V_canon = build_krylov_basis(Z0, phi, m=4)

print(f"Canonical Z_0 operator (diagonal):")
print(Z0.real.astype(int))
print(f"\nKrylov basis dimension: {V_canon.shape[1]}  (full space = 4)")
print(f"-> Diagonal operators do NOT span full space")

### Why Canonical is Different

The diagonal operator $Z_0$ gives Krylov dimension **1 out of 4**.

- $Z_0|00\rangle = +1 \cdot |00\rangle$ (eigenstate!)
- Repeated application: $H^n|00\rangle = |00\rangle$ for all $n$
- Krylov subspace = span{$|00\rangle$} only

**Diagonal operators preserve the computational basis**, so the Krylov subspace doesn't grow. This is why Canonical Krylov shows meaningful phase transitions.

## Scaling Analysis: How Fast Does Krylov Span Full Space?

We now compare how quickly the Krylov dimension grows with K (number of Hamiltonian operators) for both models at d=8.

**What to look for:**
- **QubitGrid**: Should reach dim/d = 1.0 quickly (full space)
- **Canonical**: Should grow slowly and possibly stay below 1.0

The plot shows the **normalized Krylov dimension** (dim/d) averaged over 20 random Hamiltonian samples at each K value.

In [ ]:
def measure_krylov_dim(model, K, phi, n_samples=20):
    """Average Krylov dimension over random submodels and lambda."""
    dims = []
    for seed in range(n_samples):
        sub = model.sample_submodel(K, seed=seed)
        rng = np.random.default_rng(seed + 1000)
        lambdas = rng.uniform(-1, 1, K)
        H = sum(l * h for l, h in zip(lambdas, sub.basis))
        V = build_krylov_basis(H, phi, m=model.dim)
        dims.append(V.shape[1])
    return np.mean(dims)

d = 8
phi = np.zeros(d, dtype=complex); phi[0] = 1.0

canonical = CanonicalQuditModel(dim=d, seed=42)
qubitgrid = QubitGridModel(dim=d, nx=1, ny=3, seed=42)

K_vals = list(range(2, 20))
canon_dims = [measure_krylov_dim(canonical, K, phi) / d for K in K_vals]
qg_dims = [measure_krylov_dim(qubitgrid, K, phi) / d for K in K_vals]

plt.figure(figsize=(8, 5))
plt.plot(K_vals, canon_dims, 'bo-', label='Canonical', markersize=6)
plt.plot(K_vals, qg_dims, 'rs-', label='QubitGrid', markersize=6)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Full space')
plt.xlabel('K (number of operators)')
plt.ylabel('dim(Krylov) / d')
plt.title(f'Krylov Subspace Dimension (d={d})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig('../fig/krylov_dimension_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"QubitGrid reaches full dimension by K={K_vals[next(i for i,v in enumerate(qg_dims) if v >= 1.0)]}")
print(f"Canonical at K={K_vals[-1]}: dim/d = {canon_dims[-1]:.2f}")

### Plot Analysis

**QubitGrid (red squares):** Reaches full dimension (dim/d = 1.0) by K ~ 9. Generic Pauli operators are "ergodic" -- they quickly mix all basis states. Once dim = d, the Krylov criterion becomes uninformative.

**Canonical (blue circles):** Grows more slowly. At K = 19, still at dim/d ~ 0.99. The structured (diagonal) operators create invariant subspaces, so the Krylov criterion remains informative for detecting unreachable states.

**Implication:** For QubitGrid with K >= 10, essentially ALL random targets will have Krylov score = 1.0, giving P(unreachable) = 0.

## Summary & Conclusion

### The Key Finding

**QubitGrid Krylov is always P=0 because generic Pauli Hamiltonians span the full Hilbert space via Krylov iteration.** This is physically correct behavior, not a bug or limitation of our code.

### Why This Happens

| Property | QubitGrid (Pauli) | Canonical |
|----------|-------------------|-----------|
| Operator type | Generic tensor products | Includes diagonal ops |
| Mixing behavior | Highly mixing | Preserves subspaces |
| Krylov growth | Fast (full by K~10) | Slow (often < d) |
| P(unreachable) | Always 0 | Shows phase transition |

### The Physics

The Krylov criterion tests: *"Can target $\psi$ be reached by time evolution under H?"*

For generic Pauli Hamiltonians:
1. The operators couple all basis states (no selection rules)
2. Time evolution explores the entire Hilbert space
3. Any target can be approximated arbitrarily well
4. **All states are genuinely reachable** -- the criterion correctly reports this

### Comparison with Other Criteria

| Criterion | What it tests | QubitGrid result |
|-----------|---------------|------------------|
| **Krylov** | Can $\psi$ be in time-evolution orbit? | P=0 (all reachable) |
| **Spectral** | Can spectral distributions match? | Phase transition |
| **Moment** | Do conservation laws forbid $\psi$? | Phase transition |

The Spectral and Moment criteria detect unreachability through **different physics** (eigenvalue distribution constraints and moment conservation laws) which remain informative even when Krylov is not.

### Conclusion

The QubitGrid Krylov result P=0 reflects the **physical reality** that generic Pauli Hamiltonians provide full controllability. For publication, we correctly **omit QubitGrid Krylov** from our plots and focus on Spectral and Moment criteria, which provide meaningful phase transition information.